# STEER Quick Start

This notebook provides a structured, first-user-friendly introduction to the core STEER workflow.
It is organized to make the logic, required inputs, and expected outputs easier to follow.

Use this notebook if your input `.h5ad` already contains the layers needed for RNA velocity analysis.
If your data does not yet include `spliced` and `unspliced`, please start from the raw-data preprocessing tutorials instead.

## What This Notebook Covers

By the end of this notebook, you will have:

- loaded and checked a prepared spatial transcriptomics dataset
- preprocessed the data into a STEER-ready representation
- run the first training stage to learn a stable cell-state representation
- derive kinetic priors and use them in the second training stage for kinetic learning
- generated velocity-related outputs and spatial visualizations

This notebook is designed to clarify the main workflow, not to expose every optional feature.

## Expected Input

The recommended input is an `AnnData` object with the following structure:

- `adata.layers['spliced']`: required
- `adata.layers['unspliced']`: required
- `adata.obs_names` and `adata.var_names`: required
- `adata.obsm['X_spatial']`: required for the spatial workflow used here

This quick start uses the demo dataset included in the repository.

In [ ]:
import os
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import seaborn as sns
import torch
import torch.backends.cudnn as cudnn

import steer
from steer.prior.prior import PriorInferenceManager

warnings.filterwarnings("ignore")
%matplotlib inline

print("=== STEER Quick Start Environment ===")
print(f"PyTorch version: {torch.__version__}")
print(f"Scanpy version: {sc.__version__}")
print(f"scVelo version: {scv.__version__}")
print(f"STEER version: {getattr(steer, '__version__', 'dev build')}")
print("====================================")

## Configuration

This section defines the dataset path, output directory, random seed, and a small set of core parameters.
The defaults below are suitable for running the bundled example and provide a reasonable starting point for most users.

In [ ]:
class Config:
    # Data
    data_name = "bilinear"
    input_dir = "tutorials/demo_data"
    result_dir = "tutorials/results"
    seed = 2026

    # Model and graph settings
    expert = 3
    smooth_neigh = 30
    spatial_neighbors = 8
    npc = 30

    # Training settings
    pretrain_epochs = 500
    finetune_epochs = 5000

    # Advanced settings
    graph = "union"
    corr_mode = "u"
    neighbor_metric = "cosine"
    use_us = True
    use_filter = False
    fine_method = "hierarchical"
    target_size = 300

cfg = Config()

INPUT_FILE = os.path.join(cfg.input_dir, f"{cfg.data_name}.h5ad")
RESULT_PATH = os.path.join(cfg.result_dir, f"{cfg.data_name}_quickstart")
os.makedirs(RESULT_PATH, exist_ok=True)

print(f"Input file: {INPUT_FILE}")
print(f"Output directory: {RESULT_PATH}")

## How To Choose Key Parameters

A few parameters strongly affect the behavior of the workflow:

- `expert`: the number of kinetic experts. In this demo we use `3` as a practical default. In real analyses, a better strategy is to estimate a candidate range from the preprocessed embedding and then use `mclust` together with entropy-based merging behavior to identify an elbow point.
- `smooth_neigh`: the number of neighbors used for smoothing. If the data quality is poor or noisy, increasing this value can help stabilize the signal. In practice, `30` or `100` often works well in many settings.
- `spatial_neighbors`: the number of neighbors used in the spatial graph. For relatively coarse-resolution data, `8` is often enough. For high-resolution data, increasing it to around `30` can improve robustness to local noise.
- `use_filter`: whether to filter genes before the kinetic-learning stage. `False` means the model keeps the full gene set after basic QC. This is often convenient for an initial run. If enabled, STEER uses its prior-inference strategy to retain genes that are more suitable for kinetic inference.

The expert-number selection workflow described below is currently shown as a documented procedure. It can be further integrated into a direct Python-facing helper in the future.

## Optional: Estimating The Expert Number

A practical way to choose `expert` is:

1. start from the preprocessed embedding stored in `adata.obsm`, such as `X_pca_combined`
2. export that embedding for model-based clustering
3. run `mclust` over a range of cluster numbers
4. inspect the entropy-based merging behavior and choose an elbow point as the expert number

An example Python snippet for exporting the embedding is shown below.

```python
import scanpy as sc
import pandas as pd

adata = sc.read_h5ad("/path/to/your_preprocessed_data.h5ad")
X_pre_embed = adata.obsm["X_pca_combined"]
df_embed = pd.DataFrame(X_pre_embed, index=adata.obs_names)
df_embed.to_csv("/path/to/X_pre_embed.csv")
```

You can then evaluate candidate expert numbers in R with `mclust` and inspect the entropy elbow.

```r
library(mclust)

df_embed <- read.csv("/path/to/X_pre_embed.csv", row.names = 1)
mresult <- Mclust(data = df_embed, G = 2:10, modelNames = "EEE", seed = 618)
output <- clustCombi(mresult, seed = 618)
combiOptim <- clustCombiOptim(output, reg = 2, plot = TRUE)

# Save the entropy elbow figure if needed
pdf("/path/to/Cluster_Number_Selection.pdf", width = 6, height = 6)
combiOptim <- clustCombiOptim(output, reg = 2, plot = TRUE)
title(main = "Cluster Number Selection")
dev.off()
```

This procedure is especially useful when the number of kinetic regimes is not obvious from the biology in advance.

## Reproducibility And Device Setup

To keep the tutorial reproducible, we fix the random seed before loading data and training the model.
The notebook will automatically use a GPU if one is available.

In [ ]:
def setup_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True
    cudnn.benchmark = False

setup_seed(cfg.seed)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load The Demo Dataset

We begin by loading the bundled demo dataset and checking whether the required layers and spatial coordinates are present.
These checks are useful when you later replace the demo file with your own data.

In [ ]:
adata = sc.read_h5ad(INPUT_FILE)

print(adata)
print("\nAvailable layers:", list(adata.layers.keys()))
print("Available obsm keys:", list(adata.obsm.keys()))
print(f"Number of cells: {adata.n_obs}")
print(f"Number of genes: {adata.n_vars}")

required_layers = ["spliced", "unspliced"]
missing_layers = [layer for layer in required_layers if layer not in adata.layers]
if missing_layers:
    raise ValueError(f"Missing required layers: {missing_layers}")
if "X_spatial" not in adata.obsm:
    raise ValueError("This quick start expects spatial coordinates in adata.obsm['X_spatial'].")

## Basic Preprocessing

`scvelo.pp.filter_and_normalize` prepares the input for downstream velocity modeling.
After that, `steer.preprocess_anndata_spatial` builds the STEER-ready dataframe, adjacency matrix, and processed `AnnData` object used by the graph-based model.

In [ ]:
scv.pp.filter_and_normalize(adata, min_shared_counts=None, n_top_genes=None, enforce=True)

df, adjacency_matrix, adata = steer.preprocess_anndata_spatial(
    adata,
    npc=cfg.npc,
    NUM_AD_NEIGH=30,
    spatial_neighbors=cfg.spatial_neighbors,
    SMOOTH_NEIGH=cfg.smooth_neigh,
    moments_adj=True,
    combine_mode=cfg.graph,
    neighbor_metric=cfg.neighbor_metric,
    spatial_key="X_spatial",
    use_us=cfg.use_us,
)

print(df.head())
print("\nAdjacency matrix shape:", adjacency_matrix.shape)

## Prepare PyTorch Geometric Inputs

STEER trains on a graph representation of the dataset.
This step packages the processed dataframe and adjacency matrix into the PyTorch Geometric format expected by the training functions.

In [ ]:
dataset = steer.preload_datasets_all_genes_anndata(df=df, MODEL_MODE="pretrain", adata=adata)
pyg_data = steer.create_pyg_data(dataset, adjacency_matrix, normalize=True)

print(pyg_data)

## Step 1: First-Stage Training

The first training stage learns an initial latent representation, expert assignments, and cell-state structure.
This stage provides the geometric foundation used by the later kinetic-learning stage.

In [ ]:
result_adata = steer.model_training_share_neighbor_adata(
    device=device,
    device2=device,
    pyg_data=pyg_data,
    MODEL_MODE="pretrain",
    adata=adata,
    NUM_LOSS_NEIGH=30,
    corr_mode=cfg.corr_mode,
    max_n_cluster=cfg.expert,
    expert_mode="slim",
    pretrain_epochs=cfg.pretrain_epochs,
    cluster_epochs=200,
    path=RESULT_PATH,
)

## Optional: Initial Clustering With `mclust`

If R and `rpy2` are available, STEER can use `mclust` for an initial clustering step.
If that environment is not available, this notebook will continue without it.

In [ ]:
try:
    result_adata = steer.mclust_R(result_adata, num_cluster=cfg.expert)
    print("Initial clustering with mclust completed.")
except Exception as e:
    print("mclust step skipped.")
    print(f"Reason: {e}")

torch.cuda.empty_cache()

## Step 2: Prior Inference

This stage derives kinetic priors from the representation learned above.
It includes defining fine clusters, optionally filtering genes, and estimating convexity-based directional information.

In [ ]:
prior_manager = PriorInferenceManager(result_adata, df, RESULT_PATH, seed=cfg.seed)
prior_manager.task1_define_fine_clusters(method=cfg.fine_method, target_size=cfg.target_size)
prior_manager.task2_filter_genes(based_on="expert", keep_ngene=1000, use_filter=cfg.use_filter)
prior_manager.task3_calc_convexity(based_on="fine_cluster")

result_adata, df_updated, fine_clus_vec_np = prior_manager.finalize_for_training()
print("Prior inference completed.")

## Step 3: Build The Kinetic-Learning Dataset

After prior inference, we subset to velocity genes and transfer the learned outputs needed for the next training stage.
We then rerun preprocessing on the dataset used for kinetic learning.

In [ ]:
prior_adata = result_adata.copy()
velo_adata = adata[:, prior_adata.var["is_velocity_gene"]].copy()

velo_adata.layers["pred_cell_type"] = prior_adata.layers["pred_cell_type"]
velo_adata.obsm["X_pre_embed"] = prior_adata.obsm["X_pre_embed"]
velo_adata.obs["pred_cluster"] = prior_adata.obs["pred_cluster"].astype(int)

df_fine, adjacency_matrix_fine, velo_adata = steer.preprocess_anndata_spatial(
    velo_adata,
    npc=cfg.npc,
    NUM_AD_NEIGH=30,
    spatial_neighbors=cfg.spatial_neighbors,
    SMOOTH_NEIGH=cfg.smooth_neigh,
    moments_adj=True,
    neighbor_metric=cfg.neighbor_metric,
    combine_mode=cfg.graph,
    spatial_key="X_spatial",
    use_us=cfg.use_us,
)

dataset_fine = steer.preload_datasets_all_genes_anndata(df=df_fine, MODEL_MODE="whole", adata=velo_adata)
pyg_data_fine = steer.create_pyg_data(dataset_fine, adjacency_matrix_fine, normalize=True)
pyg_data_fine.fine_clus_vec = torch.tensor(fine_clus_vec_np, dtype=torch.long, device=device)

print(velo_adata)
print("Velocity-gene subset shape:", velo_adata.shape)

## Step 4: Kinetic Learning With Guided Priors

This is the main kinetic-learning stage in STEER.
The model now learns kinetic behavior using the prior information estimated above, which helps disentangle kinetic regimes and improve downstream interpretability.

In [ ]:
velo_adata = steer.model_training_share_neighbor_adata(
    device=device,
    device2=device,
    pyg_data=pyg_data_fine,
    MODEL_MODE="whole",
    adata=velo_adata,
    NUM_LOSS_NEIGH=30,
    max_n_cluster=cfg.expert,
    corr_mode=cfg.corr_mode,
    pretrain_epochs=500,
    cluster_epochs=200,
    velo_batch_size=512,
    MIN_IMPRO=0.01,
    PATIENCE=100,
    num_epochs=cfg.finetune_epochs,
    path=RESULT_PATH,
)

print("Kinetic-learning stage completed.")

## Step 5: Post-Processing And Velocity Field Construction

Once kinetic learning is complete, we normalize the predicted velocity layers, clean the output object, compute neighbors in the refined embedding space, and construct the velocity graph used for visualization.

In [ ]:
result_adata = velo_adata.copy()
result_adata = steer.normalize_l2_anndata(result_adata, layer_vu="pred_vu", layer_vs="pred_vs")
result_adata = steer.clean_anndata(result_adata)

sc.pp.neighbors(result_adata, use_rep="X_refine_embed", n_neighbors=100)
steer.velocity_graph(result_adata, vkey="pred_vs_norm", xkey="model_Ms")

print(result_adata)

## Inspect The Main Outputs

At this point, the returned `AnnData` object should contain the main STEER outputs needed for downstream analysis.
The checks below are helpful when you start applying the workflow to your own data.

In [ ]:
print("obs columns:")
print(sorted(result_adata.obs.columns.tolist()))

print("\nobsm keys:")
print(sorted(result_adata.obsm.keys()))

print("\nlayers:")
print(sorted(result_adata.layers.keys()))

## Visualization

Here we visualize the refined velocity field in spatial coordinates, colored by expert assignment and latent time.
This is often the most intuitive first check that the pipeline ran successfully.

In [ ]:
scv.settings.figdir = RESULT_PATH
scv.set_figure_params(style="scvelo", dpi=150, figsize=(5, 4), transparent=True)

scv.pl.velocity_embedding_stream(
    result_adata,
    basis="spatial",
    vkey="pred_vs_norm",
    color=["Expert", "Pred Time"],
    title=["Expert Assignment", "Latent Time"],
    show=True,
)

## Common Notes

- If `mclust` is unavailable, the notebook can still run, but the optional clustering step will be skipped.
- If you encounter installation issues, check the package versions for PyTorch, PyG, `torch-scatter`, and `torch-sparse` first.
- If you want to use your own data, the easiest path is to replace the demo `.h5ad` with a file that already contains `spliced`, `unspliced`, and spatial coordinates.

## Next Steps

After finishing this notebook, you can continue with:

- the figure notebooks if you want to reproduce paper analyses
- the raw-data preprocessing tutorials if your data does not yet contain `spliced` and `unspliced`
- your own prepared `.h5ad` data, as long as it contains `spliced`, `unspliced`, and spatial coordinates for the workflow used here